# 04 — Language Understanding and Generation

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Summarize articles, search by meaning, enhance content, and expand queries.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### Extractive summarization
Sentences ranked by TF-IDF salience; top ones returned in order.

In [2]:
from src.language_models.summarizer import Summarizer
summ = Summarizer(method="extractive")
art = df["text"].iloc[0]
print(summ.summarize(art, n_sentences=2))

Ad sales boost Time Warner profit Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier. But its film division saw profits slump 27% to $284m, helped by box-office flops Alexander and Catwoman, a sharp contrast to year-earlier, when the third and final film in the Lord of the Rings trilogy boosted results.


### Semantic search
Rank the corpus by meaning, not keywords.

In [3]:
from src.language_models.embeddings import SemanticSearch
search = SemanticSearch(backend="tfidf").index(df["text"])
for i, score, snippet in search.search("football championship final", k=3):
    print(score, df["category"].iloc[i], "-", snippet[:70])

0.217 sport - Legendary Dutch boss Michels dies Legendary Dutch coach Rinus Michels,
0.181 tech - Football Manager scores big time For the past decade or so the virtual
0.124 sport - England claim Dubai Sevens glory England beat Fiji 26-21 in a dramatic


### Content enhancement and query expansion

In [4]:
from src.language_models.generator import ContentGenerator
from src.newsbot import NewsBot
bot = NewsBot().train(df["text"], df["category"])
a = bot.analyze("Apple unveiled a new AI chip, challenging Nvidia.")
print(ContentGenerator().enhance(a))
print("\nExpanded:", ContentGenerator.expand_query("market crash"))

This article reads as tech news with a positive tone. Key entities include Apple, AI, Nvidia.

Expanded: clang clangor crash market marketplace wreck


**Takeaway.** Light defaults (extractive summaries, TF-IDF search) work with no heavy downloads; transformer and SBERT upgrades are available behind a flag.